## Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## Import Package

In [ ]:
import pandas as pd
import numpy as np

## Load Raw Data

In [ ]:
recall_df = pd.read_csv('/content/drive/MyDrive/Milestone II/Data/df_recalls_raw.csv', low_memory=False)
complaints_df = pd.read_csv('/content/drive/MyDrive/Milestone II/Data/df_complaints_raw.csv', low_memory=False)
complaints_df = complaints_df.dropna(subset=['YEARTXT'])

recall_df['YEARTXT'] = recall_df['YEARTXT'].astype(int)
recall_df["POTAFF"] = pd.to_numeric(recall_df["POTAFF"], errors="coerce")
recall_df["RCDATE"] = pd.to_datetime(recall_df["RCDATE"], format="%Y%m%d", errors="coerce")

complaints_df['YEARTXT'] = complaints_df['YEARTXT'].astype(int)
complaints_df['DATEA'] = pd.to_datetime(complaints_df['DATEA'], format="%Y%m%d", errors="coerce")
complaints_df['LDATE'] = pd.to_datetime(complaints_df['LDATE'], format="%Y%m%d", errors="coerce")
complaints_df['FAILDATE'] = pd.to_datetime(complaints_df['DATEA'], format="%Y%m%d", errors="coerce")

print(f"Recall dataset size: {recall_df.shape}")
print(f"Complaint dataset size: {complaints_df.shape}")

Recall dataset size: (245553, 29)
Complaint dataset size: (2249816, 51)


In [ ]:
recall_df.head()

,RECORD_ID,CAMPNO,MAKETXT,MODELTXT,YEARTXT,MFGCAMPNO,COMPNAME,MFGNAME,BGMAN,ENDMAN,...,DESC_DEFECT,CONEQUENCE_DEFECT,CORRECTIVE_ACTION,NOTES,RCL_CMPT_ID,MFR_COMP_NAME,MFR_COMP_DESC,MFR_COMP_PTNO,DO_NOT_DRIVE,PARK_OUTSIDE
0,81716,10V341000,STERLING,9500,2008,10C-5,ENGINE AND ENGINE COOLING,DAIMLER TRUCKS NORTH AMERICA,20060905.0,20100630.0,...,DAIMLER TRUCKS IS RECALLING CERTAIN MODEL YEAR...,A SHORT CIRCUITED GRID HEATER COULD RESULT IN ...,DAIMLER TRUCKS WILL NOTIFY OWNERS AND DEALERS ...,DAIMLER TRUCKS' RECALL CAMPAIGN NUMBER IS 10C-...,000036542000755786000000127,NaN,NaN,NaN,No,No
1,81717,10V341000,STERLING,ACTERRA,2009,10C-5,ENGINE AND ENGINE COOLING,DAIMLER TRUCKS NORTH AMERICA,20060905.0,20100630.0,...,DAIMLER TRUCKS IS RECALLING CERTAIN MODEL YEAR...,A SHORT CIRCUITED GRID HEATER COULD RESULT IN ...,DAIMLER TRUCKS WILL NOTIFY OWNERS AND DEALERS ...,DAIMLER TRUCKS' RECALL CAMPAIGN NUMBER IS 10C-...,000036542000779778000000127,NaN,NaN,NaN,No,No
2,81718,10V341000,FCCC,S2,2011,10C-5,ENGINE AND ENGINE COOLING,DAIMLER TRUCKS NORTH AMERICA,20060905.0,20100630.0,...,DAIMLER TRUCKS IS RECALLING CERTAIN MODEL YEAR...,A SHORT CIRCUITED GRID HEATER COULD RESULT IN ...,DAIMLER TRUCKS WILL NOTIFY OWNERS AND DEALERS ...,DAIMLER TRUCKS' RECALL CAMPAIGN NUMBER IS 10C-...,000036542000873063000000127,NaN,NaN,NaN,No,No
3,81719,10V341000,FREIGHTLINER,M2,2011,10C-5,ENGINE AND ENGINE COOLING,DAIMLER TRUCKS NORTH AMERICA,20060905.0,20100630.0,...,DAIMLER TRUCKS IS RECALLING CERTAIN MODEL YEAR...,A SHORT CIRCUITED GRID HEATER COULD RESULT IN ...,DAIMLER TRUCKS WILL NOTIFY OWNERS AND DEALERS ...,DAIMLER TRUCKS' RECALL CAMPAIGN NUMBER IS 10C-...,000036542000875770000000127,NaN,NaN,NaN,No,No
4,81720,10V341000,FREIGHTLINER,FLD,2007,10C-5,ENGINE AND ENGINE COOLING,DAIMLER TRUCKS NORTH AMERICA,20060905.0,20100630.0,...,DAIMLER TRUCKS IS RECALLING CERTAIN MODEL YEAR...,A SHORT CIRCUITED GRID HEATER COULD RESULT IN ...,DAIMLER TRUCKS WILL NOTIFY OWNERS AND DEALERS ...,DAIMLER TRUCKS' RECALL CAMPAIGN NUMBER IS 10C-...,000036542000748863000000127,NaN,NaN,NaN,No,No


In [ ]:
complaints_df.head()

,CMPLID,ODINO,MFR_NAME,MAKETXT,MODELTXT,YEARTXT,CRASH,FAILDATE,FIRE,INJURED,...,DEALER_TEL,DEALER_CITY,DEALER_STATE,DEALER_ZIP,PROD_TYPE,REPAIRED_YN,MEDICAL_ATTN,VEHICLES_TOWED_YN,STATE_OF_INCIDENT,VEHICLE_OPERATOR
0,1,958241,"Volvo Car USA, LLC",VOLVO,760,1987,N,1995-01-03,N,0,...,NaN,NaN,NaN,NaN,V,NaN,N,N,NaN,NaN
1,2,958130,Ford Motor Company,FORD,THUNDERBIRD,1992,N,1995-01-03,N,0,...,NaN,NaN,NaN,NaN,V,NaN,N,N,NaN,NaN
2,3,958132,"Kia America, Inc.",KIA,SEPHIA,1994,Y,1995-01-03,N,0,...,NaN,NaN,NaN,NaN,V,NaN,N,N,NaN,NaN
3,4,958133,"Chrysler (FCA US, LLC)",DODGE,600,1987,N,1995-01-03,N,0,...,NaN,NaN,NaN,NaN,V,NaN,N,N,NaN,NaN
4,5,958137,"Chrysler (FCA US, LLC)",DODGE,CARAVAN,1991,N,1995-01-03,N,0,...,NaN,NaN,NaN,NaN,V,NaN,N,N,NaN,NaN


## Cleaning

In [ ]:
# RCLTYPECD: Vehicle, Equipment, Child Restraint, Or Tire
# Only keeping Vehicle type
recall_df = recall_df[recall_df['RCLTYPECD'] == 'V'].copy()
print("After dropping non-Vehicle type campaigns:", recall_df['CAMPNO'].nunique())

After dropping non-Vehicle type campaigns: 13642


In [ ]:
def clean_key(s):
  return s.str.strip().str.upper()

def count_prior_complaints(row):
  key = (row['mk'], row['md'], row['yr'])
  if key not in complaint_dates.index or pd.isna(row['RCDATE']):
    return 0
  dates = complaint_dates.loc[key]
  return int(np.searchsorted(dates, np.datetime64(row['RCDATE']), side='left'))

for d in (recall_df, complaints_df):
  d['mk'] = clean_key(d['MAKETXT'])
  d['md'] = clean_key(d['MODELTXT'])
  d['yr'] = d['YEARTXT']

first_complaint = (
  complaints_df.groupby(['mk', 'md', 'yr'])['DATEA'].min()
  .rename('first_complaint_date').reset_index()
)

joined = recall_df.merge(first_complaint, on=['mk', 'md', 'yr'], how='left')
joined['has_prior_complaint'] = joined['first_complaint_date'].notna() & (joined['first_complaint_date'] < joined['RCDATE'])

complaint_dates = (
  complaints_df.dropna(subset=['DATEA'])
  .sort_values('DATEA')
  .groupby(['mk', 'md', 'yr'])['DATEA']
  .apply(lambda s: s.values)
)

joined['n_prior_complaints'] = joined.apply(count_prior_complaints, axis=1)

campaign_joinable = joined.groupby('CAMPNO')['has_prior_complaint'].any()
print("Total vehicle campaigns:", len(campaign_joinable))
print("Campaigns with >=1 prior complaint on exact make/model/year key:", campaign_joinable.sum(), f"({campaign_joinable.mean():.1%})")


Total vehicle campaigns: 13642
Campaigns with >=1 prior complaint on exact make/model/year key: 6180 (45.3%)


In [ ]:
# Drop campaigns that have no prior complaint at all
joinable_campnos = campaign_joinable[campaign_joinable].index

print("Campaigns before dropping:", recall_df['CAMPNO'].nunique())
recall_df = recall_df[recall_df['CAMPNO'].isin(joinable_campnos)].copy()
joined = joined[joined['CAMPNO'].isin(joinable_campnos)].copy()
print("Campaigns after dropping (have >=1 prior complaint):", recall_df['CAMPNO'].nunique())
print("Recall rows remaining:", recall_df.shape)


Campaigns before dropping: 13642
Campaigns after dropping (have >=1 prior complaint): 6180
Recall rows remaining: (163828, 32)


## Target Label Grouping

Further grouping by following: [Car Components Explained: Exploring Car Parts & Their Functions](https://www.uti.edu/blog/automotive/car-components)

In [ ]:
def group_PARENT_COMPNAME(df, source_col='PARENT_COMPNAME', target_col='PARENT_GROUP'):
  category_mapping = {
    # Group 1: Engine
    "ENGINE AND ENGINE COOLING": "Engine",
    "ENGINE": "Engine",
    "HYBRID PROPULSION SYSTEM": "Engine",
    "FUEL/PROPULSION SYSTEM": "Engine",

    # Group 2: Electrical
    "ELECTRICAL SYSTEM": "Electrical",
    "ELECTRONIC STABILITY CONTROL (ESC)": "Electrical",
    "VEHICLE SPEED CONTROL": "Electrical",
    "TRACTION CONTROL SYSTEM": "Electrical",

    # Group 3: Brakes
    "SERVICE BRAKES, HYDRAULIC": "Brakes",
    "SERVICE BRAKES, AIR": "Brakes",
    "PARKING BRAKE": "Brakes",
    "SERVICE BRAKES": "Brakes",
    "SERVICE BRAKES, ELECTRIC": "Brakes",
    "SERVICE BRAKES, HYDRAULIC; AUTOHOLD BRAKE SYSTEM/BRAKE HOLD": "Brakes",

    # Group 4: Transmission
    "POWER TRAIN": "Transmission",

    # Group 5: Chassis
    "STEERING": "Chassis",
    "SUSPENSION": "Chassis",

    # Group 6: Fuel
    "FUEL SYSTEM, GASOLINE": "Fuel",
    "FUEL SYSTEM, DIESEL": "Fuel",
    "FUEL SYSTEM, OTHER": "Fuel",

    # Group 7: Tires
    "TIRES": "Tires",
    "WHEELS": "Tires",

    # Group 8: Structure
    "STRUCTURE": "Structure",

    # Group 9: Equipment
    "EQUIPMENT": "Equipment",
    "EQUIPMENT ADAPTIVE/MOBILITY": "Equipment",

    # Group 10: Communication
    "COMMUNICATION": "Communication",

    # Group 11: Restraints
    "AIR BAGS": "Restraints",
    "SEAT BELTS": "Restraints",

    # Group 12: ADAS
    "BACK OVER PREVENTION": "ADAS",
    "FORWARD COLLISION AVOIDANCE": "ADAS",
    "LANE DEPARTURE": "ADAS",

    # Group 13: Vision
    "EXTERIOR LIGHTING": "Vision",
    "VISIBILITY": "Vision",
    "VISIBILITY/WIPER": "Vision",
    "INTERIOR LIGHTING": "Vision",

    # Group 14: Seating
    "SEATS": "Seating",
    "CHILD SEAT": "Seating",

    # Group 15: Hardware
    "LATCHES/LOCKS/LINKAGES": "Hardware",
    "TRAILER HITCHES": "Hardware",

    # Group 16: Other
    "UNKNOWN OR OTHER": "Other",
    "OTHER": "Other"
  }

  # Fill 'Unclassified' to categories are not in the rule
  df[target_col] = df[source_col].map(category_mapping).fillna("Unclassified")


In [ ]:
recall_df['PARENT_COMPNAME'] = recall_df['COMPNAME'].str.split(':').str[0].str.strip()
group_PARENT_COMPNAME(recall_df)
recall_df[['COMPNAME', 'PARENT_COMPNAME', 'PARENT_GROUP']]

,COMPNAME,PARENT_COMPNAME,PARENT_GROUP
0,ENGINE AND ENGINE COOLING,ENGINE AND ENGINE COOLING,Engine
1,ENGINE AND ENGINE COOLING,ENGINE AND ENGINE COOLING,Engine
2,ENGINE AND ENGINE COOLING,ENGINE AND ENGINE COOLING,Engine
3,ENGINE AND ENGINE COOLING,ENGINE AND ENGINE COOLING,Engine
4,ENGINE AND ENGINE COOLING,ENGINE AND ENGINE COOLING,Engine
...,...,...,...
245506,SEAT BELTS,SEAT BELTS,Restraints
245507,SEAT BELTS,SEAT BELTS,Restraints
245533,STRUCTURE:BODY,STRUCTURE,Structure
245545,EQUIPMENT:OTHER:OWNERS/SERVICE/OTHER MANUAL,EQUIPMENT,Equipment


In [ ]:
recall_df.groupby('PARENT_GROUP')['CAMPNO'].nunique().sort_values(ascending=False)

,CAMPNO
PARENT_GROUP,
Electrical,1026
Restraints,938
Equipment,724
Vision,632
Chassis,566
Brakes,521
Transmission,428
Fuel,412
Engine,396


In [ ]:
# Apply the SAME category mapping to Complaints' own component field (COMPDESC)
complaints_df['COMPDESC_top'] = complaints_df['COMPDESC'].str.split(':').str[0].str.strip()
group_PARENT_COMPNAME(complaints_df, source_col='COMPDESC_top', target_col='PARENT_GROUP')
complaints_df[['COMPDESC_top', 'PARENT_GROUP']].head()

,COMPDESC_top,PARENT_GROUP
0,ENGINE AND ENGINE COOLING,Engine
1,"FUEL SYSTEM, GASOLINE",Fuel
2,POWER TRAIN,Transmission
3,"FUEL SYSTEM, GASOLINE",Fuel
4,SEATS,Seating


In [ ]:
complaints_df.groupby('PARENT_GROUP').size().sort_values(ascending=False)

,0
PARENT_GROUP,
Electrical,358765
Engine,354955
Chassis,234575
Transmission,219351
Brakes,198523
Restraints,195984
Vision,143249
Other,118241
Tires,106019


In [ ]:
complaints_df[complaints_df['PARENT_GROUP'] == 'Unclassified'][['COMPDESC_top', 'PARENT_GROUP', 'CDESCR']].head(20)


,COMPDESC_top,PARENT_GROUP,CDESCR
73367,ELECTRONIC STABILITY CONTROL,Unclassified,SWITCH UNDER THE HOOD THAT IS CONNECTED TO LIG...
85284,ELECTRONIC STABILITY CONTROL,Unclassified,FUEL GAUGE MALFUNCTION CAUSED BY FAILED DASH U...
102116,NaN,Unclassified,NO SUMMARY LISTED FOR ABOVE VEHICLE. *AK
106523,NaN,Unclassified,NO SUMMARY
109089,NaN,Unclassified,NO SUMMARY
109370,NaN,Unclassified,NO SUMMARY
109378,NaN,Unclassified,NO SUMMARY
111829,NaN,Unclassified,I BELIEVE THAT THIS MODEL OF THE CAR WAS RECAL...
113637,NaN,Unclassified,NO SUMMARY
116307,NaN,Unclassified,NO SUMMARY


## Save Preprocess Result

In [ ]:
# Some CAMPNO has more than 1 PARENT_GROUP label
recall_df[recall_df['CAMPNO']=="12V025000"][['CAMPNO', 'MAKETXT', 'MODELTXT', 'YEARTXT', 'PARENT_COMPNAME', 'PARENT_GROUP']]

,CAMPNO,MAKETXT,MODELTXT,YEARTXT,PARENT_COMPNAME,PARENT_GROUP
11243,12V025000,HONDA,RIDGELINE,2011,EQUIPMENT,Equipment
11244,12V025000,HONDA,RIDGELINE,2011,TIRES,Tires
11245,12V025000,HONDA,RIDGELINE,2012,EQUIPMENT,Equipment
11246,12V025000,HONDA,RIDGELINE,2012,TIRES,Tires


In [ ]:
# Option 1: drop multi-label campaigns
n_groups_per_campaign = recall_df.groupby('CAMPNO')['PARENT_GROUP'].nunique()
single_label_campnos = n_groups_per_campaign[n_groups_per_campaign == 1].index

recall_df_option1 = recall_df[recall_df['CAMPNO'].isin(single_label_campnos)].copy()
print("Option 1 - single-label only:")
print("  campaigns kept:", recall_df_option1['CAMPNO'].nunique(), "of", recall_df['CAMPNO'].nunique())

recall_df_option1.to_csv('/content/drive/MyDrive/Milestone II/Data/df_recalls_single_label.csv', index=False)


Option 1 - single-label only:
  campaigns kept: 5528 of 6180


In [ ]:
# Option 1 result: multi-label campaigns are gone
recall_df_option1[recall_df_option1['CAMPNO']=="12V025000"][['CAMPNO', 'MAKETXT', 'MODELTXT', 'YEARTXT', 'PARENT_COMPNAME', 'PARENT_GROUP']]

,CAMPNO,MAKETXT,MODELTXT,YEARTXT,PARENT_COMPNAME,PARENT_GROUP


In [ ]:
# Option 2: keep multi-label campaigns as is
recall_df_option2 = recall_df.copy()
print("Option 2 - all campaigns kept (including multi-label):")
print("  campaigns kept:", recall_df_option2['CAMPNO'].nunique(), "of", recall_df['CAMPNO'].nunique())

recall_df_option2.to_csv('/content/drive/MyDrive/Milestone II/Data/df_recalls_keep_multilabel.csv', index=False)


Option 2 - all campaigns kept (including multi-label):
  campaigns kept: 6180 of 6180


In [ ]:
# Option 2 result: multi-label campaigns are kept
recall_df_option2[recall_df_option2['CAMPNO']=="12V025000"][['CAMPNO', 'MAKETXT', 'MODELTXT', 'YEARTXT', 'PARENT_COMPNAME', 'PARENT_GROUP']]

,CAMPNO,MAKETXT,MODELTXT,YEARTXT,PARENT_COMPNAME,PARENT_GROUP
11243,12V025000,HONDA,RIDGELINE,2011,EQUIPMENT,Equipment
11244,12V025000,HONDA,RIDGELINE,2011,TIRES,Tires
11245,12V025000,HONDA,RIDGELINE,2012,EQUIPMENT,Equipment
11246,12V025000,HONDA,RIDGELINE,2012,TIRES,Tires


In [ ]:
# Saving processed complaint dataset
complaints_df.to_csv('/content/drive/MyDrive/Milestone II/Data/df_complaints_categorized.csv', index=False)
print("Saved categorized complaints:", complaints_df.shape)

Saved categorized complaints: (2249816, 56)


## Testing

In [ ]:
recall_df[recall_df['RCDATE'] < '2000-01-01']

,RECORD_ID,CAMPNO,MAKETXT,MODELTXT,YEARTXT,MFGCAMPNO,COMPNAME,MFGNAME,BGMAN,ENDMAN,...,DESC_DEFECT,CONEQUENCE_DEFECT,CORRECTIVE_ACTION,NOTES,RCL_CMPT_ID,MFR_COMP_NAME,MFR_COMP_DESC,MFR_COMP_PTNO,DO_NOT_DRIVE,PARK_OUTSIDE


In [ ]:
complaints_df[(complaints_df['MAKETXT'] == 'HYUNDAI')&(complaints_df['MODELTXT'] == 'AZERA')&(complaints_df['YEARTXT'] == 2006)&(complaints_df['DATEA'] < '2010-01-08')][['MAKETXT', 'MODELTXT', 'YEARTXT', 'COMPDESC', 'CDESCR']]

,MAKETXT,MODELTXT,YEARTXT,COMPDESC,CDESCR
584573,HYUNDAI,AZERA,2006,VEHICLE SPEED CONTROL,DT*: THE CONTACT STATED WHILE MAKING A LEFT T...
586280,HYUNDAI,AZERA,2006,AIR BAGS,PASSENGER AIR BAG INTERMITTENTLY DISENGAGES WH...
587868,HYUNDAI,AZERA,2006,EXTERIOR LIGHTING:HEADLIGHTS,DT: THE CONTACT STATED 30 FEET AHEAD OF THE VE...
592394,HYUNDAI,AZERA,2006,EXTERIOR LIGHTING:HEADLIGHTS,DT*: THE CONTACT STATED WHILE DRIVING AT ANY ...
599979,HYUNDAI,AZERA,2006,SUSPENSION:FRONT:SHOCK ABSORBER,DT*: THE CONTACT STATED WHILE TRAVELING OVER B...
...,...,...,...,...,...
739373,HYUNDAI,AZERA,2006,SEATS,SEVERAL WEEKS AGO THE DRIVERS SIDE POWER SEAT ...
740292,HYUNDAI,AZERA,2006,ENGINE AND ENGINE COOLING:ENGINE,"THROTTLE BODY, TPS SENSOR. CAR STOPS RUNNING...."
743731,HYUNDAI,AZERA,2006,ENGINE AND ENGINE COOLING,INTERMITTENT STARTING PROBLEMS WITH 2006 HYUND...
743935,HYUNDAI,AZERA,2006,EXTERIOR LIGHTING:HEADLIGHTS,I OWN A 2006 AZERA BOUGHT NEW IN FEBRUARY 2006...


In [ ]:
complaints_df[(complaints_df['MAKETXT'] == 'HYUNDAI')&(complaints_df['MODELTXT'] == 'AZERA')&(complaints_df['YEARTXT'] == 2006)&(complaints_df['DATEA'] < '2010-01-08')].iloc[0,]['CDESCR']

'DT*:  THE CONTACT STATED WHILE MAKING A LEFT TURN AT 5MPH THE VEHICLE SUDDENLY ACCELERATED WITHOUT WARNING.  AFTER RELEASING THE ACCELERATOR AND APPLYING THE BRAKE THE PROBLEM STOPPED.  THE VEHICLE WAS NOT INSPECTED.  '

In [ ]:
complaints_df[['DATEA', 'LDATE', 'FAILDATE']]

,DATEA,LDATE,FAILDATE
0,1995-01-03,1995-01-03,1995-01-03
1,1995-01-03,1995-01-03,1995-01-03
2,1995-01-03,1995-01-03,1995-01-03
3,1995-01-03,1995-01-03,1995-01-03
4,1995-01-03,1995-01-03,1995-01-03
...,...,...,...
2249835,2026-09-27,2026-09-27,2026-09-27
2249836,2026-09-27,2026-09-27,2026-09-27
2249837,2026-09-27,2026-09-27,2026-09-27
2249838,2026-09-27,2026-09-27,2026-09-27
